# EXP-016 — LR chọn đặc trưng

## 01 — Thông tin
Hồ sơ theo kế hoạch chung EXP-001–020. Phân tích kết quả chạy lại đã lưu; ngày đồng bộ 19/11/2026 không phải ngày huấn luyện. Mã nguồn cũ `THAI-EXP-004` chỉ dùng truy vết.

In [1]:
from pathlib import Path
import json
import hashlib
import yaml
import pandas as pd
from IPython.display import display, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/wesad/models.py").is_file())
CONFIG_PATH = ROOT / 'experiments/04_additional/EXP-016_LR_Features/config.yaml'
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
RESULTS_PATH = ROOT / 'experiments/04_additional/EXP-016_LR_Features/results.csv'
FOLDER = CONFIG_PATH.parent
PROVENANCE = json.loads((FOLDER / "provenance.json").read_text(encoding="utf-8"))

## 02 — Câu hỏi và giả thuyết
Chọn đặc trưng trong train có cải thiện so với LR đủ 68 đặc trưng không?
Kết quả có thể tăng hoặc giảm; không điều chỉnh thiết kế theo outer test.

## 03 — Dữ liệu
Bảng physiological68, giữ nguyên schema, checksum, nhãn và 15 subject theo cấu hình đã chạy. Không đưa metadata vào X.

In [2]:
feature_path = ROOT / CONFIG["features"]
if feature_path.is_file():
    assert hashlib.sha256(feature_path.read_bytes()).hexdigest() == CONFIG["feature_sha256"]
for name, expected in PROVENANCE["source_hashes"].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected
assert hashlib.sha256(RESULTS_PATH.read_bytes()).hexdigest() == PROVENANCE["source_hashes"][CONFIG["source_archive"] + "/results.csv"]
print("Đã đối chiếu nguồn và checksum; không huấn luyện lại.")

Đã đối chiếu nguồn và checksum; không huấn luyện lại.


## 04 — Protocol
LOSO 15 người; binary và multiclass tách riêng. Imputer/scaler học trong train. SelectKBest(f_classif), k=[16,32,48,68] chọn bằng inner GroupKFold 3 phần trong outer train.

## 05 — Cấu hình
`config.yaml` là cấu hình đọc hồ sơ theo mã chung. `training_settings` và `source_config` giữ cấu hình lịch sử với mã nguồn cũ; không sửa manifest để giả như đã train bằng mã mới.

In [3]:
display(CONFIG)

{'experiment_id': 'EXP-016', 'record_kind': 'analysis_of_saved_rerun', 'legacy_run_id': 'THAI-EXP-004', 'source_archive': 'experiments/04_additional/THAI-EXP-004_LR_Features/r02', 'source_config': 'experiments/04_additional/THAI-EXP-004_LR_Features/r02/config.yaml', 'source_notebook': 'notebooks/experiments/Thai_EXP-004_working.ipynb', 'features': 'data/features/g2_physiological68_group15_20261107/all_subjects.csv', 'schema': 'data/features/g2_physiological68_group15_20261107/feature_schema.json', 'feature_sha256': '2f2fddd678f333a800f2869d2e78b407934b2c3492190ed70e28d8a9107fd8ba', 'tasks': ['multiclass', 'binary'], 'seed': 42, 'reference_experiment_id': 'EXP-002', 'training_settings': {'experiment_id': 'THAI-EXP-004', 'group_experiment_id': 'EXP-016', 'run_no': 'r02', 'run_scope': 'main', 'features': 'data/features/g2_physiological68_group15_20261107/all_subjects.csv', 'schema': 'data/features/g2_physiological68_group15_20261107/feature_schema.json', 'feature_sha256': '2f2fddd678f333a

## 06 — Nguồn thực thi
Notebook này đọc kết quả đã lưu, mặc định không train. Code chạy thật và cấu hình nằm tại `notebooks/experiments/Thai_EXP-004_working.ipynb` và `experiments/04_additional/THAI-EXP-004_LR_Features/r02`. Notebook nguồn giữ code và output của lần chạy cũ; dùng runner chung ở `src/wesad/` cho các baseline. Không đổi tên API/module G3/G4 chỉ để đổi nhãn giai đoạn.

## 07 — Kết quả
Mean và SD (ddof=1) tính theo 15 người; không dùng điểm gộp cửa sổ thay cho mean theo subject.

In [4]:
metrics = pd.read_csv(RESULTS_PATH)
assert len(metrics) == 30 and not metrics.duplicated(["task", "subject_id"]).any()
assert metrics.groupby("task").size().to_dict() == {"binary": 15, "multiclass": 15}
display(metrics)
summary = metrics.groupby("task")[["macro_f1", "balanced_accuracy"]].agg(["mean", "std"])
display(summary)

   subject_id        task model  ...  selected_k  inner_macro_f1  seconds
0          S2  multiclass    lr  ...          32        0.709649   0.9225
1          S3  multiclass    lr  ...          16        0.703014   1.0577
2          S4  multiclass    lr  ...          16        0.736921   0.8189
3          S5  multiclass    lr  ...          32        0.679010   1.0849
4          S6  multiclass    lr  ...          32        0.680207   1.0178
5          S7  multiclass    lr  ...          48        0.651556   1.1501
6          S8  multiclass    lr  ...          16        0.664856   0.8746
7          S9  multiclass    lr  ...          32        0.668146   1.1043
8         S10  multiclass    lr  ...          32        0.711013   0.9039
9         S11  multiclass    lr  ...          32        0.684563   0.8671
10        S13  multiclass    lr  ...          16        0.669199   0.9499
11        S14  multiclass    lr  ...          48        0.676777   0.8109
12        S15  multiclass    lr  ...  

## 08 — Hình minh họa
Hình dựng từ kết quả/dự đoán đã lưu, dùng mã EXP chung. Không huấn luyện lại. Ảnh mã cũ trong thư mục chỉ dùng truy vết.

In [5]:
image_paths = sorted((FOLDER / "figures").glob(CONFIG["experiment_id"] + "_*.png"))
assert len(image_paths) == 5
for image_path in image_paths:
    display(Image(filename=str(image_path)))

<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>


## 09 — Phân tích và đối chứng
Đối chứng chính: EXP-002. Chỉ so cùng task, subject và dữ liệu.

In [6]:
comparison_path = ROOT / CONFIG["source_archive"] / "comparison_by_subject.csv"
if comparison_path.is_file():
    comparison = pd.read_csv(comparison_path)
    display(comparison)
    delta_columns = [c for c in comparison if c.endswith("_delta")]
    if delta_columns:
        display(comparison.groupby("task")[delta_columns].agg(["mean", "std"]))
for task, part in metrics.groupby("task"):
    display(part.nsmallest(3, "macro_f1")[["task", "subject_id", "macro_f1", "balanced_accuracy"]])

   subject_id        task  ...  macro_f1_delta  balanced_accuracy_delta
0          S2  multiclass  ...        0.025798                -0.027828
1          S3  multiclass  ...        0.238895                 0.268152
2          S4  multiclass  ...        0.028733                 0.016378
3          S5  multiclass  ...       -0.372958                -0.308361
4          S6  multiclass  ...       -0.005438                -0.002246
5          S7  multiclass  ...       -0.087184                -0.042398
6          S8  multiclass  ...        0.104634                 0.140673
7          S9  multiclass  ...        0.017778                 0.039259
8         S10  multiclass  ...        0.024315                 0.033923
9         S11  multiclass  ...        0.267420                 0.237266
10        S13  multiclass  ...       -0.188048                -0.154674
11        S14  multiclass  ...        0.027100                -0.005456
12        S15  multiclass  ...        0.036021                 0

## 10 — Kết luận và nguồn gốc
- binary: Macro-F1 0.878943 ± 0.148313; Balanced Accuracy 0.888401 ± 0.135539.
- multiclass: Macro-F1 0.612709 ± 0.176236; Balanced Accuracy 0.658928 ± 0.142906.

Kết quả biến động giữa người; không kết luận mô hình cuối chỉ từ một hướng. Chọn đặc trưng là nghiên cứu mở rộng sau khi đã quan sát test. Kết quả không phải kiểm định trên tập độc lập mới.

Nguồn cấu hình, checksum và commit được ghi tại `provenance.json`. So sánh train–test của EXP-001/002/011/016 ở `notebooks/04_evaluation/EXP_LR_train_test_comparison.ipynb`.